# Module 4 — Nettoyage de données : rendre un dataset exploitable

Dans le monde réel, les données sont rarement propres. Ce module apprend à repérer et corriger les problèmes les plus fréquents.

> **Objectif** : identifier valeurs manquantes, doublons, formats incorrects et valeurs aberrantes, puis produire un dataset propre.


## 1. Créer un dataset volontairement sale

On démarre avec de vraies erreurs : accents, espaces, doublons, valeurs manquantes, prix au mauvais format.


In [ ]:
import pandas as pd

ventes_sales = pd.DataFrame({
    "client": ["Awa", " Moussa ", "Fatou", "Awa", None],
    "produit": ["Formation Python", "Coaching Data", "Template Excel", "Formation Python", "Audit Data"],
    "montant": ["29000", "45000", None, "29000", "60000"],
    "statut": ["payé", "EN ATTENTE", "payé", "payé", "payé"]
})

print(ventes_sales)


     client           produit montant      statut
0       Awa  Formation Python   29000        payé
1   Moussa      Coaching Data   45000  EN ATTENTE
2     Fatou    Template Excel    None        payé
3       Awa  Formation Python   29000        payé
4      None        Audit Data   60000        payé


## 2. Diagnostiquer le dataset

Avant de nettoyer, on observe.


In [ ]:
print("Dimensions :", ventes_sales.shape)
print("Valeurs manquantes :")
print(ventes_sales.isna().sum())
print("Doublons :", ventes_sales.duplicated().sum())


Dimensions : (5, 4)
Valeurs manquantes :
client     1
produit    0
montant    1
statut     0
dtype: int64
Doublons : 1


## 3. Nettoyer les textes

Les espaces invisibles et les majuscules incohérentes créent de mauvaises catégories.


In [ ]:
df = ventes_sales.copy()

df["client"] = df["client"].str.strip()
df["statut"] = df["statut"].str.lower().str.strip()

print(df[["client", "statut"]])


   client      statut
0     Awa        payé
1  Moussa  en attente
2   Fatou        payé
3     Awa        payé
4    None        payé


## 4. Gérer les valeurs manquantes

Une valeur manquante doit être traitée selon le contexte.

| Colonne | Choix possible |
|---|---|
| client manquant | Remplacer par `Client inconnu`. |
| montant manquant | Remplacer par 0 ou exclure selon le cas. |

> **Point d'attention** : remplacer par 0 peut fausser une moyenne. Il faut savoir pourquoi on le fait.


In [ ]:
df["client"] = df["client"].fillna("Client inconnu")
df["montant"] = df["montant"].fillna("0")

print(df.isna().sum())


client     0
produit    0
montant    0
statut     0
dtype: int64


## 5. Convertir les types

Les montants étaient du texte. Pour calculer, il faut les convertir en nombre.


In [ ]:
df["montant"] = df["montant"].astype(int)

print(df["montant"].sum())
print(df.dtypes)


163000
client     object
produit    object
montant     int64
statut     object
dtype: object


## 6. Supprimer les doublons

Une ligne répétée peut gonfler artificiellement le chiffre d'affaires.


In [ ]:
avant = len(df)
df = df.drop_duplicates()
apres = len(df)

print("Lignes avant :", avant)
print("Lignes après :", apres)


Lignes avant : 5
Lignes après : 4


## 7. Contrôle qualité final

Un dataset propre doit passer des contrôles simples.


In [ ]:
print("Valeurs manquantes restantes :")
print(df.isna().sum())
print("Doublons restants :", df.duplicated().sum())
print("CA propre :", df["montant"].sum(), "FCFA")


Valeurs manquantes restantes :
client     0
produit    0
montant    0
statut     0
dtype: int64
Doublons restants : 0
CA propre : 134000 FCFA


## 8. Pipeline de nettoyage réutilisable

Un bon analyste ne nettoie pas au hasard. Il crée une fonction réutilisable.


In [ ]:
def nettoyer_ventes(df):
    df = df.copy()
    df["client"] = df["client"].str.strip().fillna("Client inconnu")
    df["statut"] = df["statut"].str.lower().str.strip()
    df["montant"] = df["montant"].fillna("0").astype(int)
    df = df.drop_duplicates()
    return df

ventes_propres = nettoyer_ventes(ventes_sales)
print(ventes_propres)


          client           produit  montant      statut
0            Awa  Formation Python    29000        payé
1         Moussa     Coaching Data    45000  en attente
2          Fatou    Template Excel        0        payé
4  Client inconnu        Audit Data    60000        payé


## Erreurs fréquentes

| Erreur | Cause | Correction |
|---|---|---|
| `astype(int)` échoue | Valeur non numérique. | Utiliser `pd.to_numeric(..., errors="coerce")`. |
| Doublons non détectés | Espaces ou casse différente. | Nettoyer texte avant `drop_duplicates()`. |
| Moyenne faussée | Valeurs manquantes remplacées par 0. | Justifier la stratégie. |
| Colonnes introuvables | Nom différent. | Afficher `df.columns`. |


## Mini-défi métier

Nettoie un dataset de prospects avec colonnes : `nom`, `telephone`, `statut`, `montant`.

Objectif :

- retirer les espaces ;
- uniformiser les statuts ;
- remplacer les montants manquants ;
- supprimer les doublons ;
- afficher le total des montants propres.

> **Validation** : tu dois expliquer chaque décision de nettoyage.
